In [1]:
import os
os.getcwd()

'/Users/sadiazaman/Downloads/Winter_25_26/group_project_12/ats_vector_rag'

In [3]:
import pandas as pd

df = pd.read_csv("data/raw/dataset.csv")
df.head()

,ID,Name,Role,Transcript,Resume,decision,Reason_for_decision,Job_Description
0,jasojo159,Jason Jones,E-commerce Specialist,"Interviewer: Good morning, Jason. It's great t...",Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,annma759,Ann Marshall,Game Developer,Interview Scene\n\nA conference room with a ta...,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,patrmc729,Patrick Mcclain,Human Resources Specialist,Interview Setting: A conference room in a medi...,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...
3,patrgr422,Patricia Gray,E-commerce Specialist,Here's a simulated professional interview for ...,Here's a professional resume for Patricia Gray...,select,Impressive leadership and communication abilit...,Be part of a passionate team at the forefront ...
4,amangr696,Amanda Gross,E-commerce Specialist,Here's the simulated interview:\n\nInterviewer...,Here's a professional resume for Amanda Gross:...,reject,Lacked leadership skills for a senior position.,We are looking for an experienced E-commerce S...


In [4]:
df.columns

Index(['ID', 'Name', 'Role', 'Transcript', 'Resume', 'decision',
       'Reason_for_decision', 'Job_Description'],
      dtype='object')

In [5]:
df.shape

(10174, 8)

In [9]:
df["decision"].value_counts(normalize=True)

decision
reject    0.502654
select    0.497346
Name: proportion, dtype: float64

In [10]:
df["Reason_for_decision"].sample(10, random_state=42).tolist()

['Needs improvement in machine learning algorithms.',
 'Insufficient system design expertise for senior role.',
 'Insufficient system design expertise for senior role.',
 'expected_experience : 0-2 years, domains: e-commerce, banking, education',
 'Perfectly aligned with data engineering needs.',
 'Highly adaptable to different work environments.',
 'experience',
 'Proven track record of achievements.',
 'Needs improvement in machine learning algorithms.',
 'Excellent full-stack development experience.']

In [11]:
df["Role"].value_counts().head(10)

Role
Data Scientist       538
Software Engineer    480
Product Manager      458
Data Engineer        447
UI Engineer          375
Data Analyst         329
data engineer        307
software engineer    307
product manager      303
data scientist       287
Name: count, dtype: int64

In [12]:
df["Resume"].str.len().describe()

count    10174.000000
mean      2884.276882
std        491.599987
min         29.000000
25%       2703.000000
50%       2920.000000
75%       3141.000000
max       4676.000000
Name: Resume, dtype: float64

In [13]:
import json

with open("data/processed/cases_stage2.jsonl") as f:
    for _ in range(3):
        case = json.loads(next(f))
        print(case["entities"])

{'skills_cv': ['data analysis', 'excel'], 'skills_jd': ['machine learning'], 'degrees': ['bachelor'], 'certifications': []}
{'skills_cv': ['github'], 'skills_jd': [], 'degrees': [], 'certifications': []}
{'skills_cv': ['data analysis'], 'skills_jd': [], 'degrees': ['bachelor'], 'certifications': []}


In [14]:
from collections import Counter
import json

skills_cv = Counter()
skills_jd = Counter()

with open("data/processed/cases_stage2.jsonl") as f:
    for line in f:
        c = json.loads(line)
        skills_cv.update(c["entities"]["skills_cv"])
        skills_jd.update(c["entities"]["skills_jd"])

print("Top CV skills:", skills_cv.most_common(10))
print("Top JD skills:", skills_jd.most_common(10))

Top CV skills: [('python', 5119), ('linux', 4469), ('github', 3621), ('agile', 3355), ('scrum', 3012), ('aws', 2921), ('machine learning', 2637), ('sql', 2419), ('azure', 2408), ('data analysis', 2331)]
Top JD skills: [('machine learning', 1089), ('python', 657), ('data science', 624), ('data analysis', 615), ('agile', 569), ('devops', 549), ('aws', 538), ('azure', 402), ('java', 392), ('cloud computing', 341)]


In [15]:
import faiss

index = faiss.read_index("data/processed/faiss_index.bin")
print(index.ntotal)
print(index.d)

10174
1536


In [16]:
import json

with open("data/processed/faiss_metadata.json") as f:
    metadata = json.load(f)

print(len(metadata))
print(metadata[0])

10174
{'case_id': 'case_00000', 'decision': 'reject', 'role': 'e-commerce specialist'}


In [17]:
import numpy as np

# pick a random index
idx = 42
vector = index.reconstruct(idx).reshape(1, -1)

scores, indices = index.search(vector, k=5)
print(indices[0])

[  42 5588 6554 7272 7709]


In [18]:
from collections import Counter

decisions = Counter(m["decision"] for m in metadata)
print(decisions)

Counter({'reject': 5114, 'select': 5060})
